# Exploratory fairness analysis

This notebook supports cautious descriptive comparisons of usability task outcomes and comprehension across interface-language groups. It does not determine whether ConsentLink is fair, accessible, or effective. Small usability rounds cannot establish statistical significance or broad fairness.

Keep the input CSV in approved restricted storage outside this repository. Never commit participant-level data, raw notes, recordings, identities, or a key linking session codes to people. Outputs are aggregate only and small groups are suppressed.

## Input format

Use one row per anonymous session per round/build with `session_code`, `round`, `build`, `language_code`, `task_1` through `task_6`, and `comprehension_purpose`, `comprehension_risk`, `comprehension_voluntary`, `comprehension_withdraw`, `comprehension_translation`.

Task values: `success`, `partial`, `not_successful`, `not_testable`, or blank. Comprehension values: `correct`, `partly_correct`, `not_understood`, `not_assessed`, or blank. Only `success` and `correct` count as positive outcomes. Blank and unavailable outcomes are excluded from that metric's denominator; they are never treated as successful.

The default minimum is five sessions in each cohort and five observed outcomes per metric. This is a disclosure guard, not evidence of statistical adequacy. Apply stricter ethics or institutional rules. Set `DATA_PATH` to an approved CSV outside the repository.

In [ ]:
import csv
from pathlib import Path

DATA_PATH = None  # Example: Path(r"C:\restricted\usability_outcomes.csv")
MIN_GROUP_N = 5
TASK_COLUMNS = [f"task_{number}" for number in range(1, 7)]
COMPREHENSION_COLUMNS = [
    "comprehension_purpose", "comprehension_risk", "comprehension_voluntary",
    "comprehension_withdraw", "comprehension_translation",
]
REQUIRED_COLUMNS = [
    "session_code", "round", "build", "language_code",
] + TASK_COLUMNS + COMPREHENSION_COLUMNS
TASK_VALUES = {"success", "partial", "not_successful", "not_testable"}
COMPREHENSION_VALUES = {
    "correct", "partly_correct", "not_understood", "not_assessed",
}

In [ ]:
def load_outcomes(path):
    rows = []
    seen = set()
    with Path(path).open(newline="", encoding="utf-8-sig") as source:
        reader = csv.DictReader(source)
        missing = set(REQUIRED_COLUMNS) - set(reader.fieldnames or [])
        if missing:
            raise ValueError("Missing required columns: " + ", ".join(sorted(missing)))
        for row in reader:
            session = (row["session_code"] or "").strip()
            round_name = (row["round"] or "").strip()
            build = (row["build"] or "").strip()
            language = (row["language_code"] or "").strip().lower()
            if not session or not round_name or not build:
                raise ValueError("Every row needs an anonymous session code, round, and build.")
            key = (session, round_name, build)
            if key in seen:
                raise ValueError("Duplicate session code within a round/build.")
            seen.add(key)
            normalized = {
                "round": round_name,
                "build": build,
                "language": language,
            }
            for column in TASK_COLUMNS:
                value = (row[column] or "").strip().lower()
                if value and value not in TASK_VALUES:
                    raise ValueError(f"Invalid category in {column}.")
                normalized[column] = value
            for column in COMPREHENSION_COLUMNS:
                value = (row[column] or "").strip().lower()
                if value and value not in COMPREHENSION_VALUES:
                    raise ValueError(f"Invalid category in {column}.")
                normalized[column] = value
            rows.append(normalized)
    return rows


def _metric_counts(rows, columns, positive_value, excluded_values):
    result = {}
    for column in columns:
        observed = [
            row[column] for row in rows
            if row[column] and row[column] not in excluded_values
        ]
        result[column] = (
            len(observed),
            sum(value == positive_value for value in observed),
        )
    return result


def compare_language_groups(rows, min_group_n=MIN_GROUP_N):
    if min_group_n < 2:
        raise ValueError("min_group_n must be at least 2.")
    results = []
    combinations = sorted({(row["round"], row["build"]) for row in rows})
    metrics = [
        (TASK_COLUMNS, "task_success_rate", "success", {"not_testable"}),
        (
            COMPREHENSION_COLUMNS,
            "comprehension_correct_rate",
            "correct",
            {"not_assessed"},
        ),
    ]
    for round_name, build in combinations:
        cohort = [
            row for row in rows
            if row["round"] == round_name and row["build"] == build
            and row["language"]
        ]
        languages = sorted({row["language"] for row in cohort})
        for language in languages:
            group = [row for row in cohort if row["language"] == language]
            others = [row for row in cohort if row["language"] != language]
            if len(group) < min_group_n or len(others) < min_group_n:
                continue
            for columns, label, positive, excluded in metrics:
                group_metrics = _metric_counts(group, columns, positive, excluded)
                other_metrics = _metric_counts(others, columns, positive, excluded)
                for metric, (group_n, group_positive) in group_metrics.items():
                    other_n, other_positive = other_metrics[metric]
                    if group_n < min_group_n or other_n < min_group_n:
                        continue
                    group_rate = group_positive / group_n
                    other_rate = other_positive / other_n
                    results.append({
                        "round": round_name,
                        "build": build,
                        "language_code": language,
                        "metric": metric,
                        "outcome": label,
                        "group_n": group_n,
                        "group_rate": round(group_rate, 3),
                        "comparison_n": other_n,
                        "comparison_rate": round(other_rate, 3),
                        "difference_percentage_points": round(
                            100 * (group_rate - other_rate), 1
                        ),
                    })
    return results


if DATA_PATH is None:
    print("Set DATA_PATH to the approved CSV outside this repository, then run this cell.")
else:
    comparisons = compare_language_groups(load_outcomes(DATA_PATH))
    if comparisons:
        print("Aggregate comparisons only; groups and metrics below the minimum are suppressed:")
        for comparison in comparisons:
            print(comparison)
    else:
        print("No group and metric met the minimum. No subgroup results are shown.")

## Interpretation and privacy

- The displayed difference is descriptive percentage points: one language group's observed rate minus the pooled rate for other known-language sessions in the same round/build. It is not causal, a statistical test, or a fairness score.
- `partial` counts in the task denominator but not as success. Only `correct` counts as a positive comprehension outcome.
- Blank, `not_testable`, and `not_assessed` values are excluded from their metric denominators, never assumed equal. Document missingness and test-build limitations in the report.
- Groups and metric denominators below the minimum are omitted. The threshold is only a disclosure guard; small usability samples are unstable and may not represent language communities.
- Do not claim statistical significance or broad fairness. Review reporting against ethics and privacy requirements.